In [1]:
import requests

url = "https://jsonplaceholder.typicode.com/users" # Public API you're pulling data from

response = requests.get(url)

print(response.status_code) # 200 confirms that the connection was successful
data = response.json() # Convert the response into usable Python data
print(data)

200
[{'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'address': {'street': 'Kulas Light', 'suite': 'Apt. 556', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'geo': {'lat': '-37.3159', 'lng': '81.1496'}}, 'phone': '1-770-736-8031 x56442', 'website': 'hildegard.org', 'company': {'name': 'Romaguera-Crona', 'catchPhrase': 'Multi-layered client-server neural-net', 'bs': 'harness real-time e-markets'}}, {'id': 2, 'name': 'Ervin Howell', 'username': 'Antonette', 'email': 'Shanna@melissa.tv', 'address': {'street': 'Victor Plains', 'suite': 'Suite 879', 'city': 'Wisokyburgh', 'zipcode': '90566-7771', 'geo': {'lat': '-43.9509', 'lng': '-34.4618'}}, 'phone': '010-692-6593 x09125', 'website': 'anastasia.net', 'company': {'name': 'Deckow-Crist', 'catchPhrase': 'Proactive didactic contingency', 'bs': 'synergize scalable supply-chains'}}, {'id': 3, 'name': 'Clementine Bauch', 'username': 'Samantha', 'email': 'Nathan@yesenia.net', 'address': {'street': 'Douglas E

In [2]:
import pandas as pd

df = pd.json_normalize(data)
print(df.columns.tolist()) # Check and confirm if the expected columns are showing on the DataFrame

['id', 'name', 'username', 'email', 'phone', 'website', 'address.street', 'address.suite', 'address.city', 'address.zipcode', 'address.geo.lat', 'address.geo.lng', 'company.name', 'company.catchPhrase', 'company.bs']


In [5]:
import pandas as pd

df = pd.json_normalize(data)
staging_df = df[[  # Select the columns you need or want to use
    "id",
    "name",
    "username",
    "email",
    "address.city",
    "address.zipcode",
    "phone",
    "website",
    "company.name"
]]
print(staging_df) 

   id                      name          username                      email  \
0   1             Leanne Graham              Bret          Sincere@april.biz   
1   2              Ervin Howell         Antonette          Shanna@melissa.tv   
2   3          Clementine Bauch          Samantha         Nathan@yesenia.net   
3   4          Patricia Lebsack          Karianne  Julianne.OConner@kory.org   
4   5          Chelsey Dietrich            Kamren   Lucio_Hettinger@annie.ca   
5   6      Mrs. Dennis Schulist  Leopoldo_Corkery    Karley_Dach@jasper.info   
6   7           Kurtis Weissnat      Elwyn.Skiles     Telly.Hoeger@billy.biz   
7   8  Nicholas Runolfsdottir V     Maxime_Nienow       Sherwood@rosamond.me   
8   9           Glenna Reichert          Delphine    Chaim_McDermott@dana.io   
9  10        Clementina DuBuque    Moriah.Stanton     Rey.Padberg@karina.biz   

     address.city address.zipcode                  phone        website  \
0     Gwenborough      92998-3874  1-770-736

In [6]:
# Rename your columns

df = staging_df.rename(columns={
    "address.city": "city",
    "address.zipcode": "postal_code",
    "company.name": "company_name"
})
print(df.columns)


Index(['id', 'name', 'username', 'email', 'city', 'postal_code', 'phone',
       'website', 'company_name'],
      dtype='object')


In [7]:
# Work on your checks, just to get an idea of what type of data you're working with
staging_df.head()
staging_df.columns
staging_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id               10 non-null     int64 
 1   name             10 non-null     object
 2   username         10 non-null     object
 3   email            10 non-null     object
 4   address.city     10 non-null     object
 5   address.zipcode  10 non-null     object
 6   phone            10 non-null     object
 7   website          10 non-null     object
 8   company.name     10 non-null     object
dtypes: int64(1), object(8)
memory usage: 852.0+ bytes


In [8]:
import urllib
from sqlalchemy import create_engine # Connect to the SQL Server and database

server = "DESKTOP-IAJP36J\\SQLEXPRESS01"   
database = "DataEngineeringPractice"

params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [9]:
from sqlalchemy import text

with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print(result.fetchone()) # Check if the connection was successful

(1,)


C:\Users\ModiehiMphuthi\AppData\Local\Temp\ipykernel_25564\675508140.py:3: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  with engine.connect() as connection:


In [11]:
df.to_sql("Client_Data_ChatGPT", con=engine, if_exists="replace", index=False)

10

In [12]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT * FROM Client_Data_ChatGPT"))
    for row in result:
        print(row)

(1, 'Leanne Graham', 'Bret', 'Sincere@april.biz', 'Gwenborough', '92998-3874', '1-770-736-8031 x56442', 'hildegard.org', 'Romaguera-Crona')
(2, 'Ervin Howell', 'Antonette', 'Shanna@melissa.tv', 'Wisokyburgh', '90566-7771', '010-692-6593 x09125', 'anastasia.net', 'Deckow-Crist')
(3, 'Clementine Bauch', 'Samantha', 'Nathan@yesenia.net', 'McKenziehaven', '59590-4157', '1-463-123-4447', 'ramiro.info', 'Romaguera-Jacobson')
(4, 'Patricia Lebsack', 'Karianne', 'Julianne.OConner@kory.org', 'South Elvis', '53919-4257', '493-170-9623 x156', 'kale.biz', 'Robel-Corkery')
(5, 'Chelsey Dietrich', 'Kamren', 'Lucio_Hettinger@annie.ca', 'Roscoeview', '33263', '(254)954-1289', 'demarco.info', 'Keebler LLC')
(6, 'Mrs. Dennis Schulist', 'Leopoldo_Corkery', 'Karley_Dach@jasper.info', 'South Christy', '23505-1337', '1-477-935-8478 x6430', 'ola.org', 'Considine-Lockman')
(7, 'Kurtis Weissnat', 'Elwyn.Skiles', 'Telly.Hoeger@billy.biz', 'Howemouth', '58804-1099', '210.067.6132', 'elvis.io', 'Johns Group')
(8,

In [13]:
df_from_sql = pd.read_sql("SELECT * FROM Client_Data_ChatGPT", con=engine)
print(df_from_sql) # Read from the SQL table to confirm if everything was pushed through

   id                      name          username                      email  \
0   1             Leanne Graham              Bret          Sincere@april.biz   
1   2              Ervin Howell         Antonette          Shanna@melissa.tv   
2   3          Clementine Bauch          Samantha         Nathan@yesenia.net   
3   4          Patricia Lebsack          Karianne  Julianne.OConner@kory.org   
4   5          Chelsey Dietrich            Kamren   Lucio_Hettinger@annie.ca   
5   6      Mrs. Dennis Schulist  Leopoldo_Corkery    Karley_Dach@jasper.info   
6   7           Kurtis Weissnat      Elwyn.Skiles     Telly.Hoeger@billy.biz   
7   8  Nicholas Runolfsdottir V     Maxime_Nienow       Sherwood@rosamond.me   
8   9           Glenna Reichert          Delphine    Chaim_McDermott@dana.io   
9  10        Clementina DuBuque    Moriah.Stanton     Rey.Padberg@karina.biz   

             city postal_code                  phone        website  \
0     Gwenborough  92998-3874  1-770-736-8031 x5